Q1

In [ ]:
from numba import cuda
import numpy as np
import time

N = 5000000

arr = np.random.random(N).astype(np.float32)

def compute(arr):
  return arr**2 +3*arr + 5

@cuda.jit
def computeNumba(arr, result):
  idx = cuda.grid(1)
  if (idx < arr.size):
    result[idx] = arr[idx]**2 + 3*arr[idx] + 5

threads = 256
blocks = (N + threads - 1)//threads


start = time.time()
new = compute(arr)
cpu_final = time.time() - start
print("Cpu time = ", cpu_final)

d_arr = cuda.to_device(arr)
d_res = cuda.device_array_like(arr)

start = time.time()
computeNumba[blocks, threads](d_arr,d_res)
cuda.synchronize()


print("Gpu time first call = ", (time.time() - start))

start = time.time()
computeNumba[blocks, threads](d_arr,d_res)
cuda.synchronize()

gpu_final = time.time() - start

print("Gpu time after startup = ", gpu_final)

print("SpeedUp : ", cpu_final/gpu_final)


Cpu time =  0.013299703598022461
Gpu time first call =  0.0714271068572998
Gpu time after startup =  0.0007827281951904297
SpeedUp :  16.99147121535181


Q2

In [3]:
import numpy as np
import time
from numba import njit


data = np.random.rand(1_000_000)
bins = 100



def hist_python(data, bins):
    hist = [0] * bins
    for val in data:
        idx = int(val * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1
    return hist



def hist_numpy(data, bins):
    return np.histogram(data, bins=bins)[0]


@njit
def hist_numba(data, bins):
    hist = np.zeros(bins, dtype=np.int32)
    for i in range(len(data)):
        val = data[i]
        idx = int(val * bins)
        if idx == bins:
            idx -= 1
        hist[idx] += 1
    return hist

start = time.time()
h1 = hist_python(data, bins)
t1 = time.time() - start

start = time.time()
h2 = hist_numpy(data, bins)
t2 = time.time() - start

# first call (compilation)
hist_numba(data, bins)

start = time.time()
h3 = hist_numba(data, bins)
t3 = time.time() - start


print("Python time:", t1)
print("NumPy time:", t2)
print("Numba time:", t3)

print("Speedups:")
print(f"Numba vs Python: {t1 / t3}x")
print(f"NumPy vs Python: {t1 / t2}x")

print("Sample Output:", h3[:10])

Python time: 0.21828913688659668
NumPy time: 0.014911413192749023
Numba time: 0.0020940303802490234
Speedups:
Numba vs Python: 104.24353865421838x
NumPy vs Python: 14.639064323745263x
Sample Output: [ 9992  9947 10016 10293  9862 10012  9944 10012 10045  9884]


Q3

In [ ]:
import numpy as np
import numba
import time
import random

def monte_carlo_pi(nsample):
  inside = 0
  for i in range(nsample):
    x = random.random()
    y = random.random()

    if (x*x + y*y < 1):
      inside += 1

  return 4*inside/nsample

@numba.njit
def monte_carlo_pi_numba(nsample):
  inside = 0
  for i in range(nsample):
    x = np.random.rand()
    y = np.random.rand()

    if (x*x + y*y < 1):
      inside += 1

  return 4*inside/nsample

N = 5000000

start = time.time()
monte_carlo_pi(N)
end_py = time.time() - start


start = time.time()
monte_carlo_pi_numba(N)
end_numba_1 = time.time() - start

start = time.time()
monte_carlo_pi_numba(N)
end_numba_2 = time.time() - start

print("Python time : ", end_py)
print("Numba wakeup call: ", end_numba_1)
print("Numba second call: ", end_numba_2)

print("Speed up : ", end_py/end_numba_2)

Python time :  1.236769676208496
Numba wakeup call:  0.13894033432006836
Numba second call:  0.05190229415893555
Speed up :  23.828805571122768


Q4

In [17]:
import numpy as np
import numba
import time

N = 10_000_000

arr = np.random.randint(0, 256, size = N, dtype=np.int32)

@numba.vectorize(['int32(int32)'])
def adjust_pixel(pixel_val):
  temp = int(pixel_val*1.2)
  if (temp > 255):
    temp = 255
  return temp

start = time.time()
result_seq = adjust_pixel(arr)
end = time.time()

print("Time on Cpu : ", end - start)

arr = np.random.randint(0, 256, size = N, dtype=np.int64)
@numba.vectorize(['int64(int64)'], target='parallel')
def adjust_pixel_parallel(pixel_val):
  temp = int(pixel_val*1.2)
  if (temp > 255):
    temp = 255
  return temp

# Warmup
result_parallel = adjust_pixel_parallel(arr)

start = time.time()
result_parallel = adjust_pixel_parallel(arr)
end_parallel = time.time()

print("Time with parallel execution on Cpu core: ", end_parallel - start)
print("Time difference: ", end-end_parallel)
print("Speedup: ", end/end_parallel)






Time on Cpu :  0.01491236686706543
Time with parallel execution on Cpu core:  0.028216123580932617
Time difference:  -0.27709388732910156
Speedup:  0.9999999998439956


Running with python list

In [20]:
arr_list = list(arr)

start = time.time()
result = adjust_pixel_parallel(arr_list)
end = time.time()

print("Time for python list: ", end - start)

Time for python list:  0.7284913063049316


Q5

In [4]:
import numpy as np
import time
from numba import njit


N = 100000
features = 10

X = np.random.randn(N, features)
y = np.random.choice([-1, 1], size=N)



def sigmoid(z):
    return 1 / (1 + np.exp(-z))



def train_numpy(X, y, lr=0.01, epochs=100):
    w = np.zeros(X.shape[1])

    for _ in range(epochs):
        z = X @ w
        preds = sigmoid(z)
        grad = X.T @ (preds - y) / len(y)
        w -= lr * grad

    return w



@njit
def train_numba(X, y, lr, epochs):
    w = np.zeros(X.shape[1])

    for _ in range(epochs):
        z = X @ w
        preds = 1 / (1 + np.exp(-z))
        grad = X.T @ (preds - y) / len(y)
        w -= lr * grad

    return w



start = time.time()
w1 = train_numpy(X, y)
t1 = time.time() - start

# first call (compilation)
train_numba(X, y, 0.01, 100)

start = time.time()
w2 = train_numba(X, y, 0.01, 100)
t2 = time.time() - start


print("NumPy time:", t1)
print("Numba time:", t2)
print("Speedup:", t1 / t2)

NumPy time: 0.22090411186218262
Numba time: 0.30657315254211426
Speedup: 0.7205592206311568


Q6



In [1]:
from numba import cuda
import numpy as np



@cuda.jit
def matAdd(a, b, c):
  idx = cuda.grid(2)
  if (idx[0] < a.shape[0] and idx[1] < a.shape[1]):
    c[idx[0], idx[1]] = a[idx[0], idx[1]] + b[idx[0], idx[1]]
rows, cols = 1024, 1024
a = np.random.randint(0, 10, size=(rows, cols))
b = np.random.randint(0, 10, size=(rows, cols))

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)

d_c = cuda.device_array_like(a)

threads = (16, 16)
blocks = ((rows + threads[0] - 1)//threads[0],
          (cols + threads[1] - 1)//threads[1])

matAdd[blocks, threads](d_a, d_b, d_c)
cuda.synchronize()

c = d_c.copy_to_host()

print("A: ", a)
print("B: ", b)
print("C: ", c)


A:  [[9 0 8 ... 2 9 3]
 [3 5 1 ... 7 2 5]
 [7 8 5 ... 8 8 9]
 ...
 [6 9 6 ... 5 1 0]
 [7 4 0 ... 4 8 4]
 [3 1 0 ... 0 9 7]]
B:  [[9 2 5 ... 7 0 4]
 [1 2 3 ... 9 0 1]
 [8 2 5 ... 5 8 6]
 ...
 [9 1 0 ... 2 3 4]
 [0 1 5 ... 4 8 8]
 [2 5 0 ... 2 3 4]]
C:  [[18  2 13 ...  9  9  7]
 [ 4  7  4 ... 16  2  6]
 [15 10 10 ... 13 16 15]
 ...
 [15 10  6 ...  7  4  4]
 [ 7  5  5 ...  8 16 12]
 [ 5  6  0 ...  2 12 11]]
